In [ ]:
!pip install -q pypdf sentence-transformers faiss-cpu transformers accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 90.8 MB/s eta 0:00:00


In [ ]:
from pypdf import PdfReader

from sentence_transformers import SentenceTransformer

import faiss
import numpy as np

from transformers import pipeline

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
reader = PdfReader("Visit-visa-checklist-Nov-2022.pdf")

pdf_text = ""

for page in reader.pages:
    pdf_text += page.extract_text()

In [ ]:
chunk_size = 500

chunks = []

for i in range(0, len(pdf_text), chunk_size):
    chunks.append(pdf_text[i:i+chunk_size])
print(chunks)
print("Number of chunks:", len(chunks))

print(chunks[0])
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)
chunk_embeddings = embedding_model.encode(chunks)
print(chunk_embeddings.shape)

dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(chunk_embeddings))


In [ ]:
chatbot = pipeline(
    "text-generation",
    model="microsoft/Phi-3-mini-4k-instruct"
)

In [ ]:

while True:

    question = input("\nAsk a question (exit to quit): ")

    if question.lower() == "exit":
        break

    # Convert question into embedding
    question_embedding = embedding_model.encode([question])

    # Search similar chunk
    distance, index_number = index.search(
        np.array(question_embedding),
        k=1
    )

    # Retrieve best chunk
    best_chunk = chunks[index_number[0][0]]
    print("Best chunk:", best_chunk)

    # Create prompt
    prompt = f"""
You are a helpful assistant.

Use ONLY the context below.

If the answer is not present, reply exactly:

I couldn't find that information.

Context:
{best_chunk}

Question:
{question}

Give ONLY the final answer.

Do NOT generate another question.
"""

    # Generate answer
    response = chatbot(
        prompt,
        max_new_tokens=20,
        do_sample=False,
        return_full_text=False
    )

    # Print answer
    print("\nBot:\n")
    print(response[0]["generated_text"])
    answer = response[0]["generated_text"]
    if "Question:" in answer:
            answer = answer.split("Question:")[0]

    print(answer.strip())